# 06 — Modèle K-Nearest Neighbors (KNN)
## Projet EDF — Prédiction de la Consommation Électrique

**Algorithme :** `KNeighborsRegressor` (scikit-learn)

Le KNN est un algorithme **instance-based** (lazy learning) : il ne construit pas de modèle explicite mais stocke tous les exemples d'entraînement. Pour prédire, il recherche les K voisins les plus proches dans l'espace des features et retourne leur consommation moyenne.

**Distance utilisée :** Euclidienne (après normalisation StandardScaler)

**Avantages :**
- Aucun apprentissage paramétrique — s'adapte aux données locales
- Capture naturellement les saisonnalités similaires (même type de jour, même saison)
- Pas d'hypothèse sur la forme de la relation

**Limitations connues :**
- Temps d'inférence élevé (O(n) comparaisons à chaque prédiction)
- Non adapté à la production temps-réel avec grands datasets

**Plan :**
1. Chargement des données
2. Optimisation du hyperparamètre K
3. Entraînement du modèle final
4. Analyse du temps d'inférence
5. Évaluation
6. Sauvegarde

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import time
from pathlib import Path

from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

from models.evaluate import evaluate_model, r2_score, mape, inference_time_ms

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 110})

PROCESSED_PATH = Path('../data/processed')
MODELS_PATH    = Path('../data/models_saved')
print('✅ Imports OK')

## 1. Chargement des données

In [ ]:
X_train = np.load(PROCESSED_PATH / 'X_train.npy')
X_test  = np.load(PROCESSED_PATH / 'X_test.npy')
y_train = np.load(PROCESSED_PATH / 'y_train.npy')
y_test  = np.load(PROCESSED_PATH / 'y_test.npy')
feature_cols = pd.read_csv(PROCESSED_PATH / 'feature_cols.csv', header=None)[0].tolist()

print(f'X_train : {X_train.shape} | y_train : {y_train.shape}')
print(f'X_test  : {X_test.shape}  | y_test  : {y_test.shape}')

## 2. Optimisation du nombre de voisins K

K est le seul hyperparamètre critique du KNN :
- **K trop petit** (ex: K=1) → sur-apprentissage (overfitting), très sensible au bruit
- **K trop grand** (ex: K=500) → sous-apprentissage (underfitting), perte de précision locale

On cherche le K optimal via validation croisée temporelle (TimeSeriesSplit).

In [ ]:
# Recherche du K optimal par validation croisée temporelle
k_range = list(range(1, 31)) + list(range(35, 76, 5))
tscv = TimeSeriesSplit(n_splits=5)

mape_train_list = []
mape_cv_list    = []

print('Optimisation de K (TimeSeriesSplit 5-fold)...')
for k in k_range:
    knn = KNeighborsRegressor(n_neighbors=k, weights='distance', metric='euclidean')
    knn.fit(X_train, y_train)
    mape_train_list.append(mape(y_train, knn.predict(X_train)))

    scores = cross_val_score(
        knn, X_train, y_train,
        cv=tscv,
        scoring='neg_mean_absolute_percentage_error'
    )
    mape_cv_list.append(-scores.mean() * 100)

optimal_k = k_range[np.argmin(mape_cv_list)]
print(f'K optimal (MAPE CV minimum) : K = {optimal_k}')
print(f'MAPE CV  à K={optimal_k} : {min(mape_cv_list):.2f}%')
print(f'MAPE train à K={optimal_k} : {mape_train_list[np.argmin(mape_cv_list)]:.2f}%')

In [ ]:
# Visualisation de la courbe de validation
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(k_range, mape_train_list, 'o-', color='steelblue',
        markersize=4, linewidth=1.5, label='MAPE Train')
ax.plot(k_range, mape_cv_list, 's-', color='coral',
        markersize=4, linewidth=1.5, label='MAPE CV (5-fold)')
ax.axvline(optimal_k, color='red', linestyle='--',
            label=f'K optimal = {optimal_k}')
ax.axhline(4.0, color='green', linestyle=':', alpha=0.7,
            label='Objectif EDF (MAPE ≤ 4%)')

ax.set_xlabel('Nombre de voisins (K)')
ax.set_ylabel('MAPE (%)')
ax.set_title('Courbe de Validation KNN — Choix du nombre de voisins K\n'
              'Compromis Biais-Variance : K petit = overfitting, K grand = underfitting',
              fontweight='bold')
ax.legend()
ax.set_ylim(0, 15)

# Zone optimale
ax.axvspan(optimal_k - 2, optimal_k + 2, alpha=0.1, color='red', label='Zone optimale')

plt.tight_layout()
plt.savefig(PROCESSED_PATH / '15_knn_validation_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nInterprétation :')
print(f'  K=1  → MAPE train ≈ 0% (mémorise tout) mais mauvaise généralisation')
print(f'  K={optimal_k} → meilleur compromis biais-variance')
print(f'  K>30 → underfitting progressif (les voisins deviennent trop dissemblables)')

## 3. Entraînement du modèle final

In [ ]:
t0 = time.time()

knn_final = KNeighborsRegressor(
    n_neighbors=optimal_k,
    weights='distance',    # Voisins proches ont plus de poids
    metric='euclidean',
    algorithm='ball_tree', # Plus efficace que 'brute' pour dim > 10
    leaf_size=30
)
knn_final.fit(X_train, y_train)

training_time = time.time() - t0
print(f'Indexation terminée en {training_time:.3f}s (construction du Ball Tree)')
print(f'K = {knn_final.n_neighbors}')
print(f'Pondération : {knn_final.weights}')
print(f'Algorithme  : {knn_final.algorithm}')
print(f'Points stockés : {len(X_train):,} (tout le jeu d\'entraînement)')

## 4. Analyse du temps d'inférence

Le KNN souffre d'un temps d'inférence élevé car chaque prédiction nécessite de calculer la distance avec tous les points d'entraînement.

In [ ]:
# Mesure du temps d'inférence selon la taille du dataset
sizes = [100, 500, 1000, len(X_train)]
inference_times = []

for size in sizes:
    knn_tmp = KNeighborsRegressor(n_neighbors=optimal_k, weights='distance',
                                   algorithm='ball_tree')
    knn_tmp.fit(X_train[:size], y_train[:size])
    times = []
    for _ in range(50):
        t = time.time()
        knn_tmp.predict(X_test[:10])
        times.append((time.time() - t) * 1000 / 10)  # ms par prédiction
    inference_times.append(np.mean(times))
    print(f'  Train size={size:,} → {inference_times[-1]:.2f} ms/prédiction')

# Inférence finale sur le modèle complet
inf_ms = inference_time_ms(knn_final, X_test)
print(f'\nTemps d\'inférence moyen (dataset complet) : {inf_ms:.2f} ms')

if inf_ms > 100:
    print('⚠️  Inférence > 100ms — KNN non recommandé pour la production temps-réel')
    print('   → Recommandation : utiliser Random Forest pour la production (< 10ms)')
else:
    print('✅ Inférence dans les limites acceptables')

# Visualisation
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sizes, inference_times, 'o-', color='steelblue', linewidth=2, markersize=8)
ax.axhline(100, color='red', linestyle='--', alpha=0.7, label='Seuil critique (100ms)')
ax.set_xlabel('Taille du jeu d\'entraînement')
ax.set_ylabel('Temps d\'inférence (ms / prédiction)')
ax.set_title('Temps d\'inférence KNN selon la taille du dataset\n'
              '(Ball Tree — complexité O(log n))', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(PROCESSED_PATH / '16_knn_inference_time.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Évaluation

In [ ]:
metrics_knn = evaluate_model(knn_final, X_test, y_test, model_name='KNN')

# Comparaison train/test
print(f'\nR² train : {r2_score(y_train, knn_final.predict(X_train)):.4f}')
print(f'R² test  : {r2_score(y_test, knn_final.predict(X_test)):.4f}')

y_pred_knn = knn_final.predict(X_test)

In [ ]:
# Visualisation des prédictions
df_proc = pd.read_parquet(PROCESSED_PATH / 'dataset_processed.parquet')
dates_test = df_proc['date'].values[len(y_train):len(y_train) + len(y_test)]

fig, axes = plt.subplots(2, 1, figsize=(15, 8))
fig.suptitle('KNN — Prédictions vs Réalité', fontweight='bold')

ax1 = axes[0]
ax1.plot(dates_test, y_test / 1000, label='Réalité', linewidth=1.2, color='steelblue')
ax1.plot(dates_test, y_pred_knn / 1000, label='KNN', linewidth=1.2,
          color='orange', linestyle='--', alpha=0.85)
ax1.set_ylabel('Consommation (GW)')
ax1.set_title('Série temporelle')
ax1.legend()

ax2 = axes[1]
ax2.scatter(y_test / 1000, y_pred_knn / 1000, alpha=0.4, s=10, color='orange')
lims = [y_test.min() / 1000 - 1, y_test.max() / 1000 + 1]
ax2.plot(lims, lims, 'r--', linewidth=1.5, label='Prédiction parfaite')
ax2.set_xlabel('Réalité (GW)'); ax2.set_ylabel('Prédiction (GW)')
ax2.set_title(f'Scatter — R²={metrics_knn["r2"]:.4f} | MAPE={metrics_knn["mape_pct"]:.2f}%')
ax2.legend()

plt.tight_layout()
plt.savefig(PROCESSED_PATH / '17_knn_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Sauvegarde

In [ ]:
model_path = MODELS_PATH / 'knn_v1.pkl'
joblib.dump(knn_final, model_path)

metrics_knn['training_time_s'] = round(training_time, 3)
metrics_knn['hyperparameters'] = {'n_neighbors': optimal_k, 'weights': 'distance',
                                    'algorithm': 'ball_tree'}
metrics_knn['inference_warning'] = 'KNN non recommandé pour production (inférence lente)'
pd.Series(metrics_knn).to_json(MODELS_PATH / 'metrics_knn.json')

print(f'✅ Modèle KNN sauvegardé → {model_path}')
print(f'\n📊 RÉSUMÉ KNN :')
print(f'   R² = {metrics_knn["r2"]}  |  MAPE = {metrics_knn["mape_pct"]}%  |  Inférence = {metrics_knn["inference_ms"]} ms')
print(f'\n⚠️  Note production : KNN conservé comme modèle de référence (baseline interprétable)')
print('   Modèle recommandé pour EDF : Random Forest (meilleur rapport MAPE/vitesse)')